# SemiRestore — KLA PS01 Colab walkthrough

This notebook is an optional reviewer and training walkthrough. KLA's required inference entry point is the standalone `run.py` script. Run the **Quick inference** section to clone, restore all public-test inputs, verify outputs, and display a sample without editing repository code. The later training cells are optional and require the organizer training archive (~919 MB).

## 1. Clone and install without replacing Colab's CUDA PyTorch


In [ ]:
from pathlib import Path
import hashlib, subprocess, sys

REPO = Path('/content/Semicon')
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
elif REPO.exists():
    raise RuntimeError(f'{REPO} exists but is not a Git checkout')
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/FaisalTabrez/Semicon.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-colab.txt')], check=True)
print('Commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short=7', 'HEAD'], text=True).strip())

## 2. Verify runtime and frozen checkpoint


In [ ]:
import platform, torch
WEIGHTS = REPO / 'models/model.pt'
expected = (REPO / 'weights/model.sha256').read_text(encoding='utf-8').split()[0]
actual = hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()
assert actual == expected, (actual, expected)
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Checkpoint SHA-256:', actual)

## 3. Quick inference on the 400 organizer public-test inputs


In [ ]:
import zipfile
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
TEST_ROOT = Path('/content/semirestore_public_test')
TEST_ZIP = TEST_ROOT / 'Test_NoisyLR.zip'
TEST_INPUT = TEST_ROOT / 'test/NoisyLR'
TEST_OUTPUT = Path('/content/semirestore_outputs')
TEST_ROOT.mkdir(parents=True, exist_ok=True)
if not TEST_ZIP.is_file():
    subprocess.run([sys.executable, '-m', 'gdown', 'https://drive.google.com/uc?id=1Ayd88_vLwVh-0of3BzL3v94DF7tTutK9', '-O', str(TEST_ZIP)], check=True)
if not TEST_INPUT.is_dir() or len(list(TEST_INPUT.glob('*.npy'))) != 400:
    root = TEST_ROOT.resolve()
    with zipfile.ZipFile(TEST_ZIP) as archive:
        for member in archive.infolist():
            if not (TEST_ROOT / member.filename).resolve().is_relative_to(root):
                raise RuntimeError(f'Unsafe archive member: {member.filename}')
        archive.extractall(TEST_ROOT)
assert len(list(TEST_INPUT.glob('*.npy'))) == 400
print('Public-test inputs:', TEST_INPUT)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
subprocess.run([sys.executable, str(REPO / 'run.py'), str(TEST_INPUT), str(TEST_OUTPUT), '--device', device, '--overwrite', '--report-json', '/content/semirestore_inference_report.json'], check=True)
subprocess.run([sys.executable, str(REPO / 'scripts/verify_submission.py'), '--input-dir', str(TEST_INPUT), '--output-dir', str(TEST_OUTPUT), '--expected-count', '400'], check=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
sample = sorted(TEST_INPUT.glob('*.npy'))[0]
restored = TEST_OUTPUT / sample.name
input_array = np.load(sample, allow_pickle=False)
output_array = np.load(restored, allow_pickle=False)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(input_array, cmap='gray'); axes[0].set_title(f'Input {input_array.shape}')
axes[1].imshow(output_array, cmap='gray'); axes[1].set_title(f'Restored {output_array.shape}')
for axis in axes: axis.axis('off')
plt.tight_layout(); plt.show()

## 4. Optional: reproduce calibration and training

The cells below use only organizer training pairs. They recreate the manifest and locked texture-held-out split, run a 200-step calibration without shortening the declared 5,000-step learning-rate schedule, and then resume to the full horizon. Skip this section when only reviewing inference.

In [ ]:
TRAIN_ROOT = Path('/content/semirestore_training')
TRAIN_ZIP = TRAIN_ROOT / 'train.zip'
TRAIN_DATA = TRAIN_ROOT / 'train'
TRAIN_ROOT.mkdir(parents=True, exist_ok=True)
if not TRAIN_ZIP.is_file():
    subprocess.run([sys.executable, '-m', 'gdown', 'https://drive.google.com/uc?id=1SNPXs_E9GHQuHiiElXOsmnzOxT4PFubx', '-O', str(TRAIN_ZIP)], check=True)
if len(list((TRAIN_DATA / 'NoisyLR').glob('*.npy'))) != 3200:
    root = TRAIN_ROOT.resolve()
    with zipfile.ZipFile(TRAIN_ZIP) as archive:
        for member in archive.infolist():
            if not (TRAIN_ROOT / member.filename).resolve().is_relative_to(root):
                raise RuntimeError(f'Unsafe archive member: {member.filename}')
        archive.extractall(TRAIN_ROOT)
MANIFEST = REPO / 'data/splits/manifest.csv'
TEXTURE_MANIFEST = REPO / 'data/splits/manifest_texture_ood.csv'
subprocess.run([sys.executable, str(REPO / 'scripts/build_manifest.py'), '--input-dir', str(TRAIN_DATA / 'NoisyLR'), '--target-dir', str(TRAIN_DATA / 'GT'), '--dataset-root', str(TRAIN_DATA), '--manifest', str(MANIFEST), '--audit', str(REPO / 'reports/dataset_audit.json'), '--expected-pairs', '3200', '--overwrite'], check=True)
subprocess.run([sys.executable, str(REPO / 'scripts/assign_texture_ood_split.py'), '--manifest', str(MANIFEST), '--dataset-root', str(TRAIN_DATA), '--output', str(TEXTURE_MANIFEST), '--audit', str(REPO / 'reports/texture_split_audit.json'), '--clusters', '12', '--validation-id-fraction', '0.15', '--validation-ood-fraction', '0.15', '--seed', '2026', '--overwrite'], check=True)
assert hashlib.sha256(TEXTURE_MANIFEST.read_bytes()).hexdigest() == '5c95b6353112e1d1ffe87f091c47af4528aff139b09fe40de9b1ffb2f030afae'
print('Locked manifest:', TEXTURE_MANIFEST)

In [ ]:
CONFIG = REPO / 'configs/final_conditioned.yaml'
CALIBRATION = Path('/content/semirestore_runs/final_calibration_200')
FINAL_RUN = Path('/content/semirestore_runs/final_conditioned_5000')
subprocess.run([sys.executable, str(REPO / 'train.py'), '--config', str(CONFIG), '--manifest', str(TEXTURE_MANIFEST), '--dataset-root', str(TRAIN_DATA), '--run-dir', str(CALIBRATION), '--device', device, '--batch-size', '16', '--num-workers', '2', '--stop-after-step', '200', '--overwrite'], check=True)
subprocess.run([sys.executable, str(REPO / 'train.py'), '--config', str(CONFIG), '--manifest', str(TEXTURE_MANIFEST), '--dataset-root', str(TRAIN_DATA), '--run-dir', str(FINAL_RUN), '--device', device, '--batch-size', '16', '--num-workers', '2', '--resume', str(CALIBRATION / 'last.pt'), '--overwrite'], check=True)
print('Reproduced checkpoint:', FINAL_RUN / 'best.pt')

## Evidence and advanced experiments

Locked metric reports are under `reports/`. The complete experiment rationale, rejected synthetic augmentation, statistics-conditioning selection, A100 execution calibration, rejected student distillation, and Carinthia cold external validation are documented in `docs/hackathon-build/` and `reports/`. Do not replace `models/model.pt` without regenerating its checksum, all 400 outputs, and every associated metric report.